In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class PatchEmbedding(nn.Module):
    # Patch Embedding: 把整张图切成 patch，并投影到统一的 embedding 维度。
    def __init__(self, img_size=224, patch_size=16, in_chans=3, embed_dim=768):
        super().__init__()
        if img_size % patch_size != 0:
            raise ValueError('img_size must be divisible by patch_size')

        self.img_size = img_size
        self.patch_size = patch_size
        # patch 总数 N = (H / P) * (W / P)
        self.num_patches = (img_size // patch_size) ** 2
        # 卷积层一步同时完成“切块 + 线性投影”，其中 kernel_size = stride = patch_size
        self.proj = nn.Conv2d(
            in_channels=in_chans,
            out_channels=embed_dim,
            kernel_size=patch_size,
            stride=patch_size,
        )

    def forward(self, x):
        # 输入形状: [B, C, H, W]
        x = self.proj(x)              # [B, E, H/P, W/P]
        x = x.flatten(2)             # [B, E, N]
        x = x.transpose(1, 2)        # [B, N, E]
        return x


# 形状检查: 224x224 的图像经过 16x16 patch 切分后，一共得到 196 个 token。
img = torch.randn(2, 3, 224, 224)
pe = PatchEmbedding()
patch_tokens = pe(img)
print('Patch Embedding Output:', patch_tokens.shape)

class MultiHeadAttention(nn.Module):
    # 多头自注意力模块，输入输出都保持 [B, N, C] 的形状。
    def __init__(self, dim, num_heads=8, attn_drop=0.0, proj_drop=0.0):
        super().__init__()
        if dim % num_heads != 0:
            raise ValueError('dim must be divisible by num_heads')

        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        # 缩放点积注意力中的缩放因子，防止点积值过大。
        self.scale = self.head_dim ** -0.5

        self.qkv = nn.Linear(dim, dim * 3)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)#这个proj是用来把多头拼接回来的结果映射回原来的维度的。因为每个 head 的输出维度是 head_dim，拼接后是 num_heads * head_dim = dim，所以这里的线性层输入输出维度都是 dim。
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x):
        B, N, C = x.shape
        # 先通过一层线性层生成 Q、K、V，然后拆成多个 head。
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        # 注意力权重 = softmax(QK^T / sqrt(d_k))
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)

        # 用注意力权重对 V 做加权求和，再把多头结果拼回去。
        out = (attn @ v).transpose(1, 2).reshape(B, N, C)
        out = self.proj(out)
        out = self.proj_drop(out)
        return out


class MLP(nn.Module):
    # Transformer block 中的前馈网络，一般把通道维扩大到 4 倍后再映射回来。
    def __init__(self, dim, mlp_ratio=4.0, drop=0.0):
        super().__init__()
        hidden_dim = int(dim * mlp_ratio)
        self.fc1 = nn.Linear(dim, hidden_dim)
        self.act = nn.GELU()
        self.drop1 = nn.Dropout(drop)
        self.fc2 = nn.Linear(hidden_dim, dim)
        self.drop2 = nn.Dropout(drop)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop1(x)
        x = self.fc2(x)
        x = self.drop2(x)
        return x


class EncoderBlock(nn.Module):
    # 一个完整的 ViT Encoder Block: LN -> MSA -> Residual -> LN -> MLP -> Residual
    def __init__(self, dim, num_heads, mlp_ratio=4.0, drop=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = MultiHeadAttention(dim, num_heads=num_heads, proj_drop=drop)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = MLP(dim, mlp_ratio=mlp_ratio, drop=drop)

    def forward(self, x):
        # 第一条残差支路: 全局 token 之间通过 self-attention 交互。
        x = x + self.attn(self.norm1(x))
        # 第二条残差支路: 每个 token 各自经过 MLP 做非线性变换。
        x = x + self.mlp(self.norm2(x))
        return x


class VisionTransformer(nn.Module):
    def __init__(
        self,
        img_size=224,
        patch_size=16,
        in_chans=3,
        num_classes=1000,
        embed_dim=768,
        depth=4,
        num_heads=8,
        mlp_ratio=4.0,
        drop=0.0,
    ):
        super().__init__()
        # 第一步: 把图像变成 patch token 序列。
        self.patch_embed = PatchEmbedding(
            img_size=img_size,
            patch_size=patch_size,
            in_chans=in_chans,
            embed_dim=embed_dim,
        )
        num_patches = self.patch_embed.num_patches

        # cls_token 用来汇聚整张图像的信息，最终拿它做分类。
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))#这个torch函数为什么用zeros？是因为在训练过程中，cls_token 会被优化器更新，所以初始值可以是零。虽然也可以用其他初始化方法，但零初始化是常见且简单的选择。
        # 位置编码长度是 num_patches + 1，因为还包含一个 cls_token。
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        self.pos_drop = nn.Dropout(drop)

        # 堆叠多个 Transformer encoder block。
        self.blocks = nn.Sequential(
            *[
                EncoderBlock(
                    dim=embed_dim,
                    num_heads=num_heads,
                    mlp_ratio=mlp_ratio,
                    drop=drop,
                )
                for _ in range(depth)
            ]
        )
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x):
        x = self.patch_embed(x)
        # 为 batch 中的每张图复制一个 cls_token。
        cls_token = self.cls_token.expand(x.shape[0], -1, -1)
        x = torch.cat((cls_token, x), dim=1)
        # 加上位置编码，否则模型无法区分 patch 的空间顺序。
        x = x + self.pos_embed
        x = self.pos_drop(x)
        x = self.blocks(x)
        x = self.norm(x)
        # 取第 0 个 token，也就是 cls_token，接分类头得到 logits。
        logits = self.head(x[:, 0])
        return logits


# 先验证 MSA 模块的输入输出形状。
msa = MultiHeadAttention(dim=768, num_heads=8)
tokens = torch.randn(2, 196, 768)
msa_out = msa(tokens)
print('MSA Output:', msa_out.shape)

# 再验证一个简化版 ViT 的整体前向传播。
vit = VisionTransformer(depth=2)
vit_out = vit(torch.randn(2, 3, 224, 224))
print('ViT Output:', vit_out.shape)

Patch Embedding Output: torch.Size([2, 196, 768])
MSA Output: torch.Size([2, 196, 768])
ViT Output: torch.Size([2, 1000])


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class PatchEmbedding(nn.Module):
    def __init__(self, image_size,patch_size,in_chans,embed_dim):
        super().__init__()
        self.image_size = image_size
        self.patch_size = patch_size
        self.num_patches = (image_size // patch_size) ** 2
        self.proj = nn.Conv2d(
                in_channels = in_chans,
                out_channels = embed_dim,
                kernel_size = patch_size,
                stride = patch_size,
        )

    def forward(self,x):
        x= self.proj(x)
        x= x.flatten(2)
        x= x.transpose(1,2)
        return x
    
class MultiHeadAttention(nn.Module):
    def __init__(self,dim,num_heads,attn_drop,proj_drop):
        super().__init__()
        self.num_heads = num_heads
        self.dim = dim
        self.head_dim = dim//num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(dim,dim*3)
        self.atten_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim,dim)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self,x):
        B,N,C = x.shape
        #加padding补 0
        if N < self.num_heads:
            pad_len = self.num_heads - N
            x = F.pad(x, (0, 0, 0, pad_len), value=0)
            N = self.num_heads
        qkv = self.qkv(x).reshape(B,N,3,self.num_heads,self.head_dim).permute(2,0,3,1,4)
        q,k,v =qkv[0],qkv[1],qkv[2]
        attn = (q @ k.transpose(-2,-1))* self.scale
        attn = attn.softmax(dim=-1)
        attn = self.atten_drop(attn)
        out = (attn@ v).transpose(1,2).reshape(B,N,C)#为什么要转置？因为在计算完注意力加权求和后，得到的形状是 [B, num_heads, N, head_dim]，需要把 num_heads 和 head_dim 拼回到一起，变成 [B, N, C] 的形状。转置是为了把 num_heads 维度放回到正确的位置。
        # B，N，C分别是什么？B 是 batch size，N 是 token 数量，C 是 embedding 维度。
        #为什么这一版不用padding补0？因为在 ViT 中，输入图像的尺寸必须是 patch_size 的整数倍，这样切分后就不会有剩余的部分，也就不需要 padding 了。如果输入图像的尺寸不是 patch_size 的整数倍，那么最后一个 patch 就会不完整，这时就需要用 padding 来补齐。
        out = self.proj(out)#reshape 之后为什么还要这一步？因为多头拼接后的维度是 num_heads * head_dim = dim，所以需要一个线性层把它映射回原来的维度。这个线性层的输入输出维度都是 dim。
        out = self.proj_drop(out)

        return out
'''传统transformer里面也有padding吗？发挥什么作用？在传统的 Transformer 中，padding 主要用于处理变长序列。因为 Transformer 的输入是一个序列
（比如文本中的单词序列），不同的样本可能有不同长度。为了让它们能够在同一批次中进行并行计算，我们需要对较短的序列进行 padding，使它们的长度一致。这样，模型就可以同时处理多个样本，而不需要单独处理每个样本的长度。
在 ViT 中，由于输入是图像，通常会要求图像的尺寸是 patch_size 的整数倍，这样切分后就不会有剩余的部分，也就不需要 padding 了。如果输入图像的尺寸不是 patch_size 的整数倍，那么最后一个 patch 就会不完整，这时就需要用 padding 来补齐。'''

class  MLP(nn.Module):
    def __init__(self,dim,mlp_ratio,drop):
        super().__init__()
        self.hidden_dim = int(dim*mlp_ratio)
        self.fc1 = nn.Linear(dim,self.hidden_dim)
        self.act = nn.GELU()
        self.drop1 = nn.Dropout(drop)
        self.fc2 = nn.Linear(self.hidden_dim,dim)
        self.drop2 = nn.Dropout(drop)

    def forward(self,x):
        x=self.fc1(x)
        x=self.act(x)
        x=self.drop1(x)
        x=self.fc2(x)
        x = self.drop2(x)

        return x
    
class EncoderBlock(nn.Module):
    def __init__(self,dim,num_heads,mlp_ratio,drop):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = MultiHeadAttention(dim,num_heads,attn_drop=drop,proj_drop=drop)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = MLP(dim,mlp_ratio,drop)

    def forward(self,x):
        x=x+self.attn(self.norm(x))
        x=self.norm(x)
        x=x+self.mlp(x)
        return x
    

class VisionTransformer(nn.Module):
    def __init__(self,dim,num_heads,attn_drop,proj_drop,mlp_ratio):
        super().__init__()
        self.patch_embed = PatchEmbedding(image_size=224,patch_size=16,in_chans=3,embed_dim=dim)
        self.cls_token = nn.Parameter(torch.zeros(1,1,dim))
        self.pos_embed = nn.Parameter(torch.zeros(1,self.patch_embed.num_patches+1,dim))#为什么patch_embed会有num这个参数？因为 patch_embed 这个模块在初始化时会计算出图像被切成 patch 后的数量，这个数量就是 num_patches。位置编码的长度需要比 num_patches 多 1，因为还要给 cls_token 留一个位置。
        self.pos_drop = nn.Dropout(proj_drop)    
        self.blocks = nn.Sequential(*[EncoderBlock(dim,num_heads,mlp_ratio,attn_drop)for _ in range(4)])
        self.norm = nn.LayerNorm(dim)

    def forward(self,x):
        x = self.patch_embed(x)
        '''为什么就能自动切分呢？怎么直接忽略了batch这个维度？
        因为在 PatchEmbedding 模块中，我们使用了一个卷积层来同时完成切块和线性投影。这个卷积层的 kernel_size 和 stride 都设置为 patch_size，这样它会自动地把输入图像切成 patch，并且每个 patch 都会被映射到一个 embedding 向量。
        卷积层的输入输出形状是 [B, C, H, W] -> [B, E, H/P, W/P]，然后我们通过 flatten 和 transpose 把它变成 [B, N, E] 的形状，其中 N 是 patch 的数量，E 是 embedding 的维度。'''
        x = torch.cat((self.cls_token.expand(x.shape[0], -1, -1), x), dim=1)
        x = x + self.pos_embed
        x = self.pos_drop(x)
        x = self.blocks(x)
        x = self.norm(x)#为什么最后还要norm？因为在 ViT 中，Transformer block 的输出会经过一个 LayerNorm 来稳定训练过程，防止梯度爆炸或消失。这个 LayerNorm 也有助于模型更好地收敛，提高性能。
        return x[:,0]#为什么取0？因为在 ViT 中，cls_token 是放在序列的第一个位置的，所以我们取 x[:, 0] 来获取 cls_token 的输出，这个输出包含了整张图像的信息，通常会被用来做分类任务。
    